In [ ]:
import os, sys
import numpy as np

import PIL
import openslide
import pickle

from tqdm import tqdm

import matplotlib.pyplot as plt

In [ ]:
import random

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)

seed = 724
set_seed(seed)

In [ ]:
relevant_path = os.getcwd()

ScannerA_WSI_Path = "Path to the WSIs dir from Scanner A"
ScannerB_WSI_Path = "Path to the WSIs dir from Scanner B" ## optional for the second scanner

extensions = ['tif', 'tiff', 'svs']
WSI_List = [fn for fn in os.listdir(ScannerA_WSI_Path) if any(fn.endswith(ext) for ext in extensions)]

Mosaic_Path = os.path.join(relevant_path, "ScannerA_HE_Mosaics")

Data = "ScannerA"

Network_Select = ["UNI", "Virchow", "Virchow2"]

In [ ]:
import torch
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, TensorDataset

## Set the batchsize
batch_size = 32

def get_embeddings(Patches, Network, patch_20x=1024):

    Data_Transform = torchvision.transforms.Compose([
        torchvision.transforms.Resize((patch_20x, patch_20x), interpolation=torchvision.transforms.InterpolationMode.BICUBIC),
        torchvision.transforms.CenterCrop((224, 224)),
        torchvision.transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    model = "Load Model Here" ## Example Load Uni Model from "https://github.com/mahmoodlab/UNI"
    
    model = model.to("cuda")
    model.eval()

    batch_tensor = torch.stack([Data_Transform(img) for img in Patches])

    
    All_Embeds = []
    for i in range(0, batch_tensor.size(0), batch_size):

        batch = batch_tensor[i:i+batch_size]
        # print(batch.shape)
        batch = batch.to("cuda")

        with torch.no_grad():
            if Network == "Virchow" or Network == "Virchow2":
                with torch.inference_mode(), torch.autocast(device_type="cuda", dtype=torch.float16):
                    Features = model(batch)
            else:    
                Features = model(batch)

        
        for i, E in enumerate(Features):
            if Network == 'Virchow':
                E = torch.unsqueeze(E, dim=0)
                class_token = E[:, 0] 
                patch_tokens = E[:, 1:]
                # concatenate class token and average pool of patch tokens
                embedding = torch.cat([class_token, patch_tokens.mean(1)], dim=-1)
                All_Embeds.append(embedding[0].cpu().detach().numpy())

            elif Network == 'Virchow2':
                E = torch.unsqueeze(E, dim=0)
                class_token = E[:, 0] 
                patch_tokens = E[:, 5:]
                # concatenate class token and average pool of patch tokens
                embedding = torch.cat([class_token, patch_tokens.mean(1)], dim=-1)
                All_Embeds.append(embedding[0].cpu().detach().numpy())

            else:
                All_Embeds.append(E.cpu().detach().numpy())


    del model, Data_Transform, batch_tensor, batch
    torch.cuda.empty_cache()
    return All_Embeds

    

# Main Mosaic Feature Generation Loop

In [ ]:
for W in WSI_List:
    W = W[:-4]
    print(W)

    with open(os.path.join(Mosaic_Path, W+"_Mosaic.pkl"), 'rb') as f:
        mosaic = pickle.load(f)

    if Data == "ScannerA":
        slide_path = os.path.join(ScannerA_WSI_Path, W+".svs")
    else:
        slide_path = os.path.join(ScannerB_WSI_Path, W+".tif")

    # Create the slide object
    slide = openslide.open_slide(slide_path)

    try:
        objective_power = int(slide.properties['openslide.objective-power'])
    except Exception as e:
        objective_power = 40.0
    # initialize the queues
    patch_queue = []
    # iterate over the patches
    for patch in tqdm(mosaic):
        # this loc is at the objective power
        h, w = patch['wsi_loc']
        # Find the patch size at 20x
        patch_size_20x = int((objective_power/20.)*1024)
        # read the patch
        patch_region = slide.read_region((w, h), 0, (patch_size_20x, patch_size_20x)).convert('RGB')
        # Add to the queue
        # patch_queue.append(np.array(patch_region)) ## Array Images
        patch_queue.append(patch_region) ## PIL images

    # patch_queue = np.array(patch_queue)
    print("Number of Patches: ", len(patch_queue))
    
    slide.close()

    for Network in Network_Select:
        print(Network)

        if os.path.exists(os.path.join(relevant_path, Data, Network, "Mosaic_Embeds")):
            print("Mosaic Embedding Path Exists")
        else:
            os.mkdir(os.path.join(relevant_path, Data, Network, "Mosaic_Embeds"))
        
        ## check if mosaic already generated or not
        if os.path.exists(os.path.join(relevant_path, Data, Network, "Mosaic_Embeds", W+"_Mosaic-Embeds.npy")):
            print("Mosaic Embeddings already Generated")
        else:
            Features = get_embeddings(patch_queue, Network)

            np.save(os.path.join(relevant_path, Data, Network, "Mosaic_Embeds", W+"_Mosaic-Embeds.npy"), Features)

            del Features
    
    
    del patch_queue
        
